In [1]:
import pandas as pd
from eutl_scraper import download
from eutl_scraper.extract import extract_projects_from_transactions

In [2]:
start = "2008-01-01"
df = pd.read_csv("data/normalized/eutl_transactions.csv", parse_dates=["transaction_date"])
df = (
    df[df.transaction_date >= pd.to_datetime(start)]
    .drop_duplicates(subset=["transaction_id"])
)

C:\Users\abrell\AppData\Local\Temp\ipykernel_10492\1551601844.py:2: DtypeWarning: Columns (20,22,23,24,25,26,27,28,29,47,49,50,51,52,53,54,55,56,60,62,65,71,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/normalized/eutl_transactions.csv", parse_dates=["transaction_date"])


In [3]:
for pref in ["transferring", "acquiring"]:
    for i in range(1, 4):
        print(f"{pref}_account_type{i}:", df[f"{pref}_account_type{i}"].unique())

transferring_account_type1: [121. 100. 120. 230.  nan 110.]
transferring_account_type2: ['121-Person Holding Account' '100-Holding Account' '-'
 '120-Operator Holding Account'
 '230-Voluntary Cancellation Account (Type 3)']
transferring_account_type3: ['0-None' '9-Aircraft Operator Account' '7-Operator Holding Account'
 '8-Person Holding Account' '12-Trading Account' '-'
 '22-International Credit Account' '1-AAU Deposit Account'
 '4-Gateway Deposit Account' '2-National Allowance Holding Account'
 '13-Auction Delivery Account' '15-Aviation Auction Account'
 '21-Aviation Allocation Account' '23-Credit Exchange Account'
 '5-Union Allowance Deletion Account'
 '6-Aviation Surrender Set-Aside Account' '14-Auction Account'
 '20-Allocation Account' '3-Central Clearing Account'
 '28-ETS AAU Deposit Account' '27-EU AAU Account'
 '31-ETS Central Clearing Account for CP2']
acquiring_account_type1: [121 100 120 230 300 250 411 210 130]
acquiring_account_type2: ['121-Person Holding Account' '100-Hol

In [4]:
pha1 = [120]
pha2 = ['8-Person Holding Account', '12-Trading Account']

pref = "transferring"
other = "acquiring"
lst_df = []
for pref, other in [("transferring", "acquiring"), ("acquiring", "transferring")]:
    df_= (
        df[df[f"{pref}_account_type1"].isin(pha1)]
        .groupby([f"{other}_account_type1", f"{other}_account_type3"], as_index=False)
        .transaction_id
        .count()
    )
    df_.columns = ["account_type1", "account_type3", "count"]
    lst_df.append(df_)
df_ = pd.concat(lst_df).groupby(["account_type1", "account_type3"], as_index=False).sum()
df_ = df_.sort_values("count", ascending=False)
df_

,account_type1,account_type3,count
1,100.0,0-None,122435
11,121.0,0-None,59767
10,120.0,0-None,34526
2,100.0,1-AAU Deposit Account,4253
6,100.0,7-Operator Holding Account,3076
12,230.0,0-None,2329
0,100.0,-,848
9,110.0,-,844
3,100.0,12-Trading Account,380
4,100.0,2-National Allowance Holding Account,125


In [5]:
df.transferring_account_type3.unique()

array(['0-None', '9-Aircraft Operator Account',
       '7-Operator Holding Account', '8-Person Holding Account',
       '12-Trading Account', '-', '22-International Credit Account',
       '1-AAU Deposit Account', '4-Gateway Deposit Account',
       '2-National Allowance Holding Account',
       '13-Auction Delivery Account', '15-Aviation Auction Account',
       '21-Aviation Allocation Account', '23-Credit Exchange Account',
       '5-Union Allowance Deletion Account',
       '6-Aviation Surrender Set-Aside Account', '14-Auction Account',
       '20-Allocation Account', '3-Central Clearing Account',
       '28-ETS AAU Deposit Account', '27-EU AAU Account',
       '31-ETS Central Clearing Account for CP2'], dtype=object)

In [7]:
transaction_columns = [
    "transaction_id",
    "transaction_type",
    "transaction_date",
    # "transaction_status",  # drop status as we anyways only observe 'completed'
    "ets_id",
    "originating_registry",
    "acquiring_registry_id",
    "acquiring_account_id",
    "acquiring_installation_id",
    "transferring_registry_id",
    "transferring_account_id",
    "transferring_installation_id",
    "unit_type_description",
    "supp_unit_type_description",
    "amount",
]
df_trans = df[transaction_columns].copy()

KeyError: "['originating_registry'] not in index"

In [ ]:
df_trans["unit_type"] = list(
    zip(df_trans.unit_type_description, df_trans.supp_unit_type_description)
)
df_trans.unit_type.unique()

array([('ERU - Emission Reduction Unit', 'No supplementary unit type'),
       ('ERU - Converted from an RMU', 'No supplementary unit type'),
       ('CER - Certified Emission Reduction Unit converted from an AAU', 'No supplementary unit type'),
       ('RMU - Removal Unit', 'No supplementary unit type'),
       ('AAU - Assigned Amount Unit', 'No supplementary unit type'),
       ('Non-Kyoto Unit', 'EU Aviation Allowances (EUAA)'),
       ('AAU - Assigned Amount Unit', 'Allowance issued for the 2008-2012 period and subsequent 5-year periods and is converted from an AAU'),
       ('Non-Kyoto Unit', 'Swiss Aviation Allowances (CHUA)'),
       ('Non-Kyoto Unit', 'Allowance issued for the 2008 to 2012 and subsequent five-year periods by a Member State that does not have AAUs'),
       ('ERU - Emission Reduction Unit', nan),
       ('tCER - Temporary CER', nan),
       ('Non-Kyoto Unit', 'EU General Allowances (EUA)'),
       ('ERU - Converted from an RMU', nan),
       ('CER - Certified Em

In [ ]:
df_trans.tail()

,transaction_id,transaction_type,transaction_date,ets_id,originating_registry,acquiring_registry_id,acquiring_account_id,acquiring_installation_id,transferring_registry_id,transferring_account_id,transferring_installation_id,unit_type_description,supp_unit_type_description,amount
2142470,EU542433,10-2,2020-01-09 18:14:20,euets,EU,EU,EU_5016380,NaN,ES,ES_5008918,ES_160,Non-Kyoto Unit,EU General Allowances (EUA),879
2142471,FR93298,10-0,2009-05-07 17:26:34,euets,PT,NaN,NaN,NaN,NaN,NaN,NaN,AAU - Assigned Amount Unit,Allowance issued for the 2008-2012 period and ...,2444
2142472,FR10339,10-0,2006-06-01 16:34:39,euets,HU,NaN,NaN,NaN,NaN,NaN,NaN,Non-Kyoto Unit,Allowance issued for the 2005-2007 period and ...,1130
2142473,EU403486,10-2,2017-04-27 13:28:12,euets,EU,EU,EU_5016380,NaN,RO,RO_5010063,RO_110,Non-Kyoto Unit,EU General Allowances (EUA),60109
2142474,EU528943,10-0,2019-08-23 13:43:04,euets,EU,HU,HU_5011776,HU_146,NL,NL_5017659,NaN,Non-Kyoto Unit,EU General Allowances (EUA),1602
